# Rotulagem de conservação dos imóveis

Gera `dados/rotulos_conservacao.csv` para treinar o classificador próprio (Fase 2).
Cada foto recebe **dois rótulos**: `label` (nível de desgaste) e `ambiente` (interna/externa).

## Labels de desgaste (olhe só o visível na foto)
- **bem conservado**: pintura/piso/banheiro aparentemente novos, sem marcas
- **desgaste leve**: marcas leves de uso, tudo funcional
- **desgastado**: desgaste evidente (pintura gasta, piso manchado, rejunte escuro)
- **precário**: mofo, infiltração, danos ou risco aparente

Em **área externa** o mesmo critério vale para fachada, muro, telhado, calçada e quintal
(pintura gasta, rachadura, telha faltando, mofo aparente). Em fachada, "mofo/infiltração"
só conta se estiver visível — não presuma.

## Ambiente (o modo persiste até você trocar)
- `i` **interna**: sala, quarto, cozinha, banheiro, corredor, área de serviço
- `e` **externa**: fachada, quintal, terreno, garagem descoberta, varanda, piscina, área comum
- `x` **ambígua**: planta, drone, mapa, foto escura/ilegível (ambiente ambíguo + desgaste ainda é dado válido)

## Comandos (digite + Enter)
- `i`/`e`/`x` define o ambiente e vale para as próximas fotos
- `1-4` rotula o desgaste com o ambiente atual | `0` pula | `z` desfaz o último e volta para a foto | `q` salva e sai
- Salvamento incremental: nada se perde se interromper; rode de novo que pula as já rotuladas
- Pode re-rodar a célula do loop sem a célula 2: fotos já rotuladas são puladas sozinhas

**Meta**: ≥80 fotos por classe de desgaste (total ~400), com pelo menos ~25% de cada ambiente
dentro de cada classe. Não mirar 80 por combinação desgaste × ambiente (seriam ~960 fotos).

Filtros da amostra: só `apartamento`/`casa`, sem fotos placeholder (`nao-encontrado`).

## Backfill
As linhas rotuladas antes de existir a coluna `ambiente` ficam com `ambiente = desconhecido`.
Rode a célula **"backfill do ambiente"** (depois do loop): ela mostra só essas fotos e pede
apenas `i`/`e`/`x`, uma tecla por foto.

In [18]:
from pathlib import Path
import csv
import os
import random
from datetime import datetime

import numpy as np
import pandas as pd

CIDADE = "joinville"
N = 800
SEED = 42
LABELS = ["bem conservado", "desgaste leve", "desgastado", "precário"]
AMBIENTES = ["interna", "externa", "ambigua", "desconhecido"]
TECLAS_AMBIENTE = {"i": "interna", "e": "externa", "x": "ambigua"}

REPO = Path.cwd()
if not (REPO / "dados").exists():
    REPO = REPO.parent  # notebook aberto a partir de Notebooks/
CSV_PATH = REPO / "dados" / "rotulos_conservacao.csv"

COLUNAS = ["url_foto", "url_anuncio", "bairro", "titulo", "valor_imovel",
           "label", "ambiente", "timestamp"]


def _normalizar(rotulo):
    """Compatível com CSV antigo (sem ambiente) e com valores inesperados."""
    r = {k: rotulo.get(k, "") for k in COLUNAS}
    amb = str(r["ambiente"] or "").strip().lower()
    r["ambiente"] = amb if amb in AMBIENTES else "desconhecido"
    return r


def _carregar_existentes():
    if not CSV_PATH.exists():
        return []
    with open(CSV_PATH, encoding="utf-8") as f:
        return [_normalizar(r) for r in csv.DictReader(f)]


def _reescrever(rotulos):
    """Grava o CSV inteiro de forma atômica (tmp + replace) para não perder rótulos."""
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    tmp = CSV_PATH.with_suffix(".csv.tmp")
    with open(tmp, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLUNAS)
        w.writeheader()
        w.writerows(_normalizar(r) for r in rotulos)
    os.replace(tmp, CSV_PATH)


def _kappa_quadratico(a, b, categorias=LABELS):
    """Kappa ponderado quadrático (a escala de desgaste é ordinal)."""
    idx = {c: i for i, c in enumerate(categorias)}
    pares = [(idx[x], idx[y]) for x, y in zip(a, b) if x in idx and y in idx]
    if not pares:
        return float("nan")
    k = len(categorias)
    obs = np.zeros((k, k))
    for i, j in pares:
        obs[i, j] += 1
    n = obs.sum()
    w = np.array([[(i - j) ** 2 for j in range(k)] for i in range(k)], dtype=float)
    w /= (k - 1) ** 2
    esperado = np.outer(obs.sum(axis=1), obs.sum(axis=0)) / n
    po, pe = (w * obs).sum() / n, (w * esperado).sum() / n
    if pe == 0:
        return 1.0 if po == 0 else float("nan")
    return 1 - po / pe

parquets = sorted((REPO / "dados" / CIDADE).glob(f"{CIDADE}_imoveis_limpo_*.parquet"))
assert parquets, f"nenhum parquet em dados/{CIDADE}"
df = pd.read_parquet(parquets[-1], columns=["url", "titulo", "bairro", "valor_imovel", "tipo_imovel", "fotos"])
print("parquet:", parquets[-1].name, "| linhas:", len(df))
df = df[df["fotos"].notna()].copy()
df = df[df["fotos"].map(lambda f: len(f) > 0)]
print("com fotos:", len(df))
# só apartamentos e casas
df = df[df["tipo_imovel"].isin(["apartamento", "casa"])].copy()
print("apto/casa:", len(df))
# fora placeholders (ex.: chavesnamao 'nao-encontrado'); mantém anúncios com ≥1 foto real
df["fotos"] = df["fotos"].map(lambda fs: [u for u in list(fs) if "nao-encontrado" not in str(u)])
df = df[df["fotos"].map(lambda f: len(f) > 0)].copy()
print("sem placeholder:", len(df))

rng = random.Random(SEED)
# 1 foto aleatória por anúncio (diversifica cômodos); ordem do df fixa p/ resume estável
df["foto"] = df["fotos"].map(lambda f: rng.choice(list(f)))

META = {"bem conservado": 110, "desgaste leve": 370, "desgastado": 295, "precário": 26}

# resume: o que já foi rotulado não volta
feitas = {r["url_foto"] for r in _carregar_existentes()}
print(f"resume: {len(feitas)} já rotuladas puladas")

# estratificada por bairro x tercil de valor; cotas sobre o que falta rotular
df["valor_num"] = pd.to_numeric(df["valor_imovel"], errors="coerce")
df["tercil"] = pd.qcut(df["valor_num"], 3, labels=["baixo", "medio", "alto"])
disp = df[~df["foto"].isin(feitas)].copy()
alvo = max(0, N - len(feitas))
celulas = disp.groupby(["bairro", "tercil"], observed=True).size()
ideal = celulas / celulas.sum() * alvo
quotas = ideal.round().astype(int).clip(lower=1)
while quotas.sum() > alvo and quotas.max() > 1:
    quotas.loc[quotas[quotas > 1].idxmax()] -= 1
for chave in (ideal - quotas).sort_values(ascending=False).index:
    if quotas.sum() >= alvo:
        break
    quotas.loc[chave] += 1

partes = []
for (bairro, tercil), qtd in quotas.items():
    g = disp[(disp["bairro"] == bairro) & (disp["tercil"] == tercil)]
    if len(g):
        partes.append(g.sample(min(len(g), int(qtd)), random_state=SEED))
amostra = pd.concat(partes).sample(frac=1, random_state=SEED)
if len(amostra) < alvo:
    usados = set(amostra["foto"])
    extra = disp[~disp["foto"].isin(usados)]
    falta = min(alvo - len(amostra), len(extra))
    if falta > 0:
        amostra = pd.concat([amostra, extra.sample(falta, random_state=SEED)])
amostra = amostra.head(N).reset_index(drop=True)

print(f"cotas: {len(celulas)} células bairro x tercil | alvo: {alvo}")
print("a rotular:", len(amostra), "| bairros:", amostra["bairro"].nunique(),
      "| tercis:", amostra["tercil"].value_counts().sort_index().to_dict())
print("dup url_foto:", int(amostra["foto"].duplicated().sum()))
amostra[["bairro", "tercil", "titulo", "valor_imovel"]].head(3)

parquet: joinville_imoveis_limpo_2026-10.parquet | linhas: 26912
com fotos: 26866
apto/casa: 23805
sem placeholder: 19301
resume: 377 já rotuladas puladas
cotas: 119 células bairro x tercil | alvo: 423
a rotular: 423 | bairros: 41 | tercis: {'baixo': 142, 'medio': 145, 'alto': 136}
dup url_foto: 0


,bairro,tercil,titulo,valor_imovel
0,centro,medio,Apartamento em Rua Duque de Caxias - Centro - ...,807265.0
1,itinga,baixo,Casa com 3 quartos à venda na Rua Waldemiro Jo...,550000.0
2,costa e silva,medio,"Casa de Condomínio com 3 Quartos à venda, 91m²...",650000.0


In [ ]:
from IPython.display import Image, clear_output, display

TECLAS = {"1": LABELS[0], "2": LABELS[1], "3": LABELS[2], "4": LABELS[3]}

# sugestões da célula auto-conservacao (funciona mesmo se ela não foi rodada)
try:
    SUGESTOES = {s["url_foto"]: s for s in _ler_csv(AUTO_CSV)
                 if not s.get("erro") and s.get("label")}
except NameError:
    SUGESTOES = {}
print(f"sugestões automáticas disponíveis: {len(SUGESTOES)}")

rotulos = _carregar_existentes()
base_len = len(rotulos)
vistos = {r["url_foto"] for r in rotulos}  # anti-duplicata: re-rodar o loop sem a célula 2 não repete fotos
indice_da_foto = {f: n for n, f in enumerate(amostra["foto"])}
modo = None  # ambiente corrente; persiste entre fotos até você trocar
salvos = 0
saiu = False
k = 0
while k < len(amostra):
    row = amostra.iloc[k]
    if row["foto"] in vistos:
        k += 1
        continue
    sug = SUGESTOES.get(row["foto"])
    clear_output(wait=True)
    display(Image(url=row["foto"], width=650))
    print(f"[{base_len + k + 1}/{base_len + len(amostra)}] {row['bairro']} | {str(row['titulo'])[:70]} | {row['valor_imovel']}")
    if modo:
        print(f"ambiente atual: {modo}")
    if sug:
        print(f"sugestão: {sug['label']} / {sug['ambiente']} (conf {sug['confianca']}) — Enter aceita")
    while True:
        t = input("Enter aceita sugestão | i/e/x ambiente | 1-4 desgaste | 0 pula | z desfaz | q sai: ").strip().lower()
        if t == "":
            if not sug:
                print("sem sugestão para esta foto — escolha ambiente e desgaste")
                continue
            rotulos.append({
                "url_foto": row["foto"], "url_anuncio": row["url"],
                "bairro": row["bairro"], "titulo": row["titulo"],
                "valor_imovel": row["valor_imovel"], "label": sug["label"],
                "ambiente": sug["ambiente"] if sug["ambiente"] in AMBIENTES else "ambigua",
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            })
            _reescrever(rotulos)
            salvos += 1
            vistos.add(row["foto"])
            k += 1
            break
        if t in TECLAS_AMBIENTE:
            modo = TECLAS_AMBIENTE[t]
            print(f"ambiente = {modo}")
            continue
        if t in TECLAS:
            if modo is None:
                print("escolha o ambiente antes (i=interna e=externa x=ambígua)")
                continue
            rotulos.append({
                "url_foto": row["foto"], "url_anuncio": row["url"],
                "bairro": row["bairro"], "titulo": row["titulo"],
                "valor_imovel": row["valor_imovel"], "label": TECLAS[t],
                "ambiente": modo,
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            })
            _reescrever(rotulos)
            salvos += 1
            vistos.add(row["foto"])
            k += 1
            break
        if t == "0":
            k += 1
            break
        if t == "z":
            if len(rotulos) > base_len:
                ultimo = rotulos.pop()
                _reescrever(rotulos)
                vistos.discard(ultimo["url_foto"])
                salvos -= 1
                if ultimo["ambiente"] in TECLAS_AMBIENTE.values():
                    modo = ultimo["ambiente"]
                k = indice_da_foto.get(ultimo["url_foto"], k)
                print(f"desfeito: {ultimo['label']} / {ultimo['ambiente']} — voltando para a foto")
            else:
                print("nada a desfazer nesta sessão")
                continue
            break
        if t == "q":
            print(f"encerrado: +{salvos} rótulos nesta sessão (total {len(rotulos)})")
            saiu = True
            k = len(amostra)
            break
        print(f"tecla inválida: {t!r}")
if not saiu:
    print(f"amostra concluída: +{salvos} rótulos (total {len(rotulos)})")

In [21]:
len(rotulos)

775

In [ ]:
import pandas as pd

if "CSV_PATH" not in globals():
    raise RuntimeError("rode a célula de configuração (cfg) antes")

rel = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
print("total:", len(rel), "| anúncios:", rel["url_anuncio"].nunique())
META = globals().get("META", {"bem conservado": 110, "desgaste leve": 370,
                              "desgastado": 295, "precário": 26})
for lab in LABELS:
    n = int((rel["label"] == lab).sum())
    print(f"  {lab:<16} {n:>3}  meta {META[lab]:>3}  faltam {max(0, META[lab] - n):>3}")
if "ambiente" in rel:
    print("ambiente:", rel["ambiente"].value_counts().to_dict())

In [ ]:
from IPython.display import Image, clear_output, display

if "_reescrever" not in globals():
    raise RuntimeError("rode a célula de configuração (cfg) antes")
try:
    rotulos
except NameError:
    rotulos = _carregar_existentes()

# só linhas rotuladas antes de existir a coluna ambiente
pendentes = [r for r in rotulos if r["ambiente"] == "desconhecido"]
print(f"{len(pendentes)} fotos sem ambiente")

continuar = True
for n, r in enumerate(pendentes):
    if not continuar:
        break
    clear_output(wait=True)
    display(Image(url=r["url_foto"], width=650))
    print(f"[{n + 1}/{len(pendentes)}] {r['bairro']} | {str(r['titulo'])[:70]} | desgaste: {r['label']}")
    while True:
        t = input("i=interna e=externa x=ambígua | 0 pula | q sai: ").strip().lower()
        if t in TECLAS_AMBIENTE:
            r["ambiente"] = TECLAS_AMBIENTE[t]
            _reescrever(rotulos)
            break
        if t == "0":
            break
        if t == "q":
            continuar = False
            break
        print(f"tecla inválida: {t!r}")

faltam = sum(1 for r in rotulos if r["ambiente"] == "desconhecido")
print(f"backfill encerrado: {len(rotulos) - faltam} com ambiente, {faltam} ainda desconhecidas")

# Rotulagem conservacao automatico

In [ ]:
# ============================================================================
# Avaliação automática v3: "vistoriador" (busca ativa de defeitos) -> desgaste
# - o modelo NÃO escolhe o label: lista achados, marca categorias, o label sai de regra fixa
# - prompt adversarial: procura problema (o prompt v1/v2 fazia o modelo responder
#   "sem manchas, sem mofo" e rotular tudo como bem conservado)
# - 800x800 + JPEG q60
# - grava dados/rotulos_conservacao_auto_v2.csv (NUNCA toca no CSV humano)
# - checkpoint a cada AUTO_SALVAR_A_CADA fotos; re-rodar pula o que já foi feito
# ============================================================================

import asyncio
import csv
import html
import os
import sys
import time
from datetime import datetime
from typing import Literal

import numpy as np
import pandas as pd
from IPython.display import HTML, display
from pydantic import BaseModel, Field

if "REPO" not in globals():
    raise RuntimeError("rode a célula de configuração (cfg) antes")

SRC = str(REPO / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

try:
    from roteador_llms.roteador_llms import LlmRouter
    from agente_avaliacao_imagens.classificacao_dgemma import fotos_para_dataurls
    import agente_avaliacao_imagens.classificacao_dgemma as _dg
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"'{e.name}' não encontrado — rode `uv pip install -e .` no kernel do notebook"
    ) from None

# ---------------------------------------------------------------- configuração
AUTO_VERSAO = "v3-vistoriador-800"
AUTO_CSV = REPO / "dados" / "rotulos_conservacao_auto_v2.csv"

AUTO_DIMENSAO = "800x800"   # resize na origem (CDN do Zap); local usa AUTO_LADO_MAX
AUTO_LADO_MAX = 800         # teto do downscale local (OLX/ChavesNaMão)
AUTO_MAX_FOTOS = 4          # fotos por anúncio (teto do helper = 4)
AUTO_CONCURRENCY = 3        # chamadas simultâneas
AUTO_TENTATIVAS = 3         # por foto
AUTO_INCLUIR_ROTULADAS = True
AUTO_SO_ROTULADAS = False   # True = roda só as fotos do CSV humano (validação)
AUTO_LIMITE = None          # ex.: 30 para conferir antes do lote inteiro
AUTO_SALVAR_A_CADA = 25     # checkpoint em disco a cada N fotos (0 desliga)
AUTO_MOSTRAR = True         # preview ao vivo da foto + veredito
AUTO_MOSTRAR_LARGURA = 420
AUTO_MODELO = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning"

AUTO_COLUNAS = COLUNAS + ["versao", "defeitos", "achados", "confianca",
                          "justificativa", "imagem_avaliavel", "modelo", "erro"]
assert 1 <= AUTO_MAX_FOTOS <= 4, "MAX_IMGS do dgemma é 4"

# resolução: o helper lê os globais do módulo a cada chamada
_dg.DIMENSAO_CDN = AUTO_DIMENSAO
_dg.LADO_MAX = AUTO_LADO_MAX

# --------------------------------------------------------------------- schema
CATEGORIAS = ["mofo", "infiltracao", "rachadura", "dano_estrutural",
              "pintura_gasta", "piso_desgastado", "manchas",
              "rejunte_escuro", "ferragens_oxidadas"]


class AchadosFoto(BaseModel):
    ambiente: Literal["interna", "externa", "ambigua"] = Field(
        description="Onde a foto foi tirada")
    imagem_avaliavel: bool = Field(
        description="false para planta, drone, mapa, render, watermark ou foto ilegível")
    achados: list[str] = Field(
        default_factory=list,
        description="0 a 5 problemas concretos com localização (ex.: 'rejunte escuro no box')")
    categorias: list[Literal["mofo", "infiltracao", "rachadura", "dano_estrutural",
                             "pintura_gasta", "piso_desgastado", "manchas",
                             "rejunte_escuro", "ferragens_oxidadas"]] = Field(
        default_factory=list,
        description="Categorias dos achados listados (vazio se nenhum)")
    justificativa: str = Field(description="1 frase sobre o estado do que está visível")
    confianca: float = Field(ge=0, le=1, description="Certeza nos achados")


GRAVES = {"mofo", "infiltracao", "rachadura", "dano_estrutural"}
FORTES = {"pintura_gasta", "piso_desgastado"}
LEVES = {"manchas", "rejunte_escuro", "ferragens_oxidadas"}


def _derivar(s):
    """Regra fixa: categorias marcadas -> label. Sem julgamento do modelo no label."""
    achados = [c for c in CATEGORIAS if c in set(s.categorias)]
    if not s.imagem_avaliavel:
        return "", achados
    if any(c in GRAVES for c in achados):
        return "precário", achados
    if any(c in FORTES for c in achados) or sum(c in LEVES for c in achados) >= 2:
        return "desgastado", achados
    if any(c in LEVES for c in achados):
        return "desgaste leve", achados
    return "bem conservado", achados


```python
PROMPT_AUTO = """Você é um avaliador de imóveis a partir de fotos de anúncio.

Classifique ESTA foto em dois eixos independentes.

<ambiente> — 
Onde a foto foi tirada:
- "interna": sala, quarto, cozinha, banheiro, corredor, área de serviço, closet
- "externa": fachada, muro, telhado, quintal, terreno, garagem descoberta, varanda, piscina, área comum
- "ambigua": planta, drone, mapa, foto escura/ilegível ou impossível decidir
<ambiente/>

<label> 
Nível de desgaste e envelhecimento aparente do imóvel (vale para interna e externa):
- "bem conservado": acabamentos aparentemente novos ou modernos, pintura uniforme, pisos e louças em boas condições, sem marcas relevantes de uso ou envelhecimento.
- "desgaste leve": pequenas marcas de uso, manchas pontuais, sinais discretos de envelhecimento, acabamentos um pouco datados, mas aparentemente bem conservados.
- "desgastado": sinais evidentes de uso ou envelhecimento, pintura gasta, pisos antigos ou manchados, rejunte escuro, ferragens oxidadas, revestimentos ultrapassados ou materiais deteriorados.
- "precário": deterioração acentuada, mofo, infiltração visível, rachaduras, partes danificadas, pintura descascada ou outros problemas relevantes de conservação.
<label/> 

<Regras>
- Julgue só o que aparece na foto. Em fachada/muro/telhado, mofo e infiltração só contam se estiverem visíveis; não presuma.
- Considere tanto o desgaste físico quanto os sinais visuais de envelhecimento dos materiais, acabamentos e elementos construtivos.
- Não confunda decoração antiga com desgaste. Considere elementos construtivos datados apenas quando houver outros sinais visuais que indiquem envelhecimento.
- Um imóvel antigo reformado, com acabamentos renovados e bem conservados, pode ser classificado como "bem conservado".
- Um imóvel aparentemente novo pode ser classificado como "desgastado" ou "precário" se apresentar sinais visíveis relevantes de deterioração.
- Não presuma a idade real do imóvel nem confunda estilo arquitetônico com deterioração. Avalie o conjunto de evidências visíveis.
- Em fotos com pouca luz, baixa resolução ou poucos detalhes, não presuma desgaste ou envelhecimento.
- Se a foto não mostrar nada avaliável (planta, drone, mapa, watermark cobrindo tudo): imagem_avaliavel=false, ambiente="ambigua" e confianca <= 0.3.
- confianca: 0 a 1, sua certeza na classificação do par (ambiente, label).
- justificativa: 1-2 frases citando o que você viu.
- evidencias: lista curta, ex.: ["piso manchado", "rejunte escuro", "esquadrias antigas"].
<Regras/>
"""

# --------------------------------------------------------------------- helpers
def _ler_csv(caminho):
    if not caminho.exists():
        return []
    with open(caminho, encoding="utf-8") as f:
        return list(csv.DictReader(f))


def _gravar_csv(caminho, linhas, colunas):
    """Escrita atômica: tmp + os.replace (mesmo padrão do _reescrever)."""
    caminho.parent.mkdir(parents=True, exist_ok=True)
    tmp = caminho.with_suffix(".csv.tmp")
    with open(tmp, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=colunas, extrasaction="ignore")
        w.writeheader()
        w.writerows(linhas)
    os.replace(tmp, caminho)


def _fotos_do_anuncio(row):
    """Foto sorteada primeiro, depois as outras do anúncio (sem repetir)."""
    vistas, saida = set(), []
    extras = row.get("fotos")
    extras = list(extras) if extras is not None else []
    for u in [row["foto"], *extras]:
        if u and u not in vistas:
            vistas.add(u)
            saida.append(u)
    return saida[:AUTO_MAX_FOTOS]


# === DEFS-AB === (o runner do A/B importa só até aqui)

# ---------------------------------------------------------------- monta a fila
feitas = {r["url_foto"] for r in _ler_csv(AUTO_CSV)
          if r.get("versao") == AUTO_VERSAO and not r.get("erro")}
fila, vistos = [], set()

if not AUTO_SO_ROTULADAS:
    for _, row in amostra.iterrows():
        for foto in _fotos_do_anuncio(row):
            if foto in feitas or foto in vistos:
                continue
            vistos.add(foto)
            fila.append({"url_foto": foto, "url_anuncio": row["url"],
                         "bairro": row["bairro"], "titulo": row["titulo"],
                         "valor_imovel": row["valor_imovel"]})

if (AUTO_INCLUIR_ROTULADAS or AUTO_SO_ROTULADAS) and CSV_PATH.exists():
    for r in _ler_csv(CSV_PATH):
        foto = r.get("url_foto", "")
        if foto and foto not in feitas and foto not in vistos:
            vistos.add(foto)
            fila.append({"url_foto": foto, "url_anuncio": r.get("url_anuncio", ""),
                         "bairro": r.get("bairro", ""), "titulo": r.get("titulo", ""),
                         "valor_imovel": r.get("valor_imovel", "")})

if AUTO_LIMITE:
    fila = fila[:int(AUTO_LIMITE)]
print(f"versão: {AUTO_VERSAO} | modelo: {AUTO_MODELO} | imagem: {AUTO_DIMENSAO}")
print(f"fila: {len(fila)} fotos | já feitas (puladas): {len(feitas)}")

# rotas: força UM modelo e desliga os demais provedores (rastreável)
ROTAS_FORCADAS = dict(groq_models=[], cerebras_models=[],
                      api_nvidia_models=[AUTO_MODELO],
                      api_langchain_nvidia_models=[], api_openai_nvidia_models=[],
                      openrouter_models=[], zai_models=[])

# ------------------------------------------------------------ classifica foto
async def _classificar(foto, sem):
    async with sem:
        for tentativa in range(1, AUTO_TENTATIVAS + 1):
            try:
                dataurls = await asyncio.to_thread(fotos_para_dataurls, [foto])
                if not dataurls:
                    raise RuntimeError("download/decodificação da foto falhou")
                conteudo = [
                    {"type": "text", "text": PROMPT_AUTO},
                    {"type": "image_url", "image_url": {"url": dataurls[0]}},
                ]
                saida = await LlmRouter(conteudo, strutured_output=AchadosFoto,
                                        **ROTAS_FORCADAS).llm_router()
                if saida is None:
                    raise RuntimeError("roteador sem resposta")
                return foto, saida
            except Exception as e:  # noqa: BLE001 - erro vira linha, não aborta
                if tentativa == AUTO_TENTATIVAS:
                    return foto, e
                await asyncio.sleep(5 * tentativa)


_preview = None
if AUTO_MOSTRAR and fila:
    try:
        _preview = display(HTML("<i>aguardando a primeira foto…</i>"), display_id=True)
    except Exception:      # frontend sem display_id: segue só com os prints
        _preview = None


def _mostrar(row, n, dt, eta):
    if _preview is None or not getattr(_preview, "update", None):
        return
    foto = html.escape(row["url_foto"], quote=True)
    if row["erro"]:
        corpo = f'<b style="color:#b00020">erro</b> {html.escape(row["erro"])}'
    else:
        corpo = (f'<b>{html.escape(row["label"] or "(não avaliável)")}</b> '
                 f'· {html.escape(row["ambiente"])} · conf {row["confianca"]}'
                 f'<br><small>{html.escape(row["defeitos"])}</small>'
                 f'<br><i>{html.escape(row["achados"])}</i>')
    try:
        _preview.update(HTML(
            f'<div><img src="{foto}" width="{AUTO_MOSTRAR_LARGURA}" '
            f'style="border-radius:6px"><br>{corpo}<br>'
            f'<small>{n}/{len(fila)} · {dt:.0f}s · ETA {eta:.0f}s</small></div>'))
    except Exception:
        pass


sem = asyncio.Semaphore(AUTO_CONCURRENCY)
lock = asyncio.Lock()                                    # serializa os checkpoints
por_url = {r["url_foto"]: r for r in _ler_csv(AUTO_CSV)
           if r.get("versao") == AUTO_VERSAO}             # base p/ retomada
feitos = 0
inicio = time.time()
checkpoints = 0


def _linha(item, saida):
    if isinstance(saida, AchadosFoto):
        label, achados = _derivar(saida)
        return {**item,
                "label": label,
                "ambiente": saida.ambiente,
                "versao": AUTO_VERSAO,
                "defeitos": "; ".join(achados),
                "achados": "; ".join(saida.achados),
                "confianca": f"{saida.confianca:.2f}",
                "justificativa": saida.justificativa,
                "imagem_avaliavel": str(saida.imagem_avaliavel).lower(),
                "modelo": AUTO_MODELO, "erro": "",
                "timestamp": datetime.now().isoformat(timespec="seconds")}
    return {**item, "label": "", "ambiente": "", "versao": AUTO_VERSAO,
            "defeitos": "", "achados": "", "confianca": "",
            "justificativa": "", "imagem_avaliavel": "",
            "modelo": AUTO_MODELO,
            "erro": f"{type(saida).__name__}: {saida}"[:200],
            "timestamp": datetime.now().isoformat(timespec="seconds")}


def _checkpoint(rotulo="", fatal=False):
    """Grava as fotos acumuladas de forma atômica."""
    try:
        _gravar_csv(AUTO_CSV, list(por_url.values()), AUTO_COLUNAS)
        print(f"  checkpoint{rotulo}: {len(por_url)} fotos em {AUTO_CSV.name}")
    except OSError as e:                      # ex.: CSV aberto no Excel no Windows
        print(f"  AVISO: checkpoint{rotulo} falhou ({e})")
        if fatal:
            raise


async def _tarefa(item):
    global feitos, checkpoints
    _, saida = await _classificar(item["url_foto"], sem)
    row = _linha(item, saida)
    async with lock:
        por_url[row["url_foto"]] = row
        feitos += 1
        dt = time.time() - inicio
        _mostrar(row, feitos, dt, dt / feitos * (len(fila) - feitos))
        if _preview is None and (feitos % 20 == 0 or feitos == len(fila)):
            print(f"  {feitos}/{len(fila)} | {dt:.0f}s | "
                  f"ETA {dt / feitos * (len(fila) - feitos):.0f}s")
        if AUTO_SALVAR_A_CADA and feitos % AUTO_SALVAR_A_CADA == 0:
            _checkpoint(f" ({feitos}/{len(fila)})")
            checkpoints += 1
    return row


if fila:
    await asyncio.gather(*[_tarefa(f) for f in fila])
_checkpoint(" (final)", fatal=True)
print(f"gravado: {AUTO_CSV.name} | {len(por_url)} linhas "
      f"({sum(1 for r in por_url.values() if r.get('erro'))} com erro)")

# ------------------------------------------------------------------ relatório
auto = pd.DataFrame(list(por_url.values()), columns=AUTO_COLUNAS)
ok = auto[(auto["erro"] == "") & (auto["label"] != "")]      # não avaliável fica fora
print("\n=== distribuição prevista ===")
print("por desgaste:")
print(ok["label"].value_counts().reindex(LABELS, fill_value=0).to_string() if len(ok) else "(vazio)")
print("\npor ambiente:")
print(ok["ambiente"].value_counts().to_string() if len(ok) else "(vazio)")

if len(ok):
    freq = pd.Series([d for hs in ok["defeitos"] for d in hs.split("; ") if d]).value_counts()
    print("\ndefeitos detectados (frequência):")
    print(freq.to_string() if len(freq) else "(nenhum)")

erros = int((auto["erro"] != "").sum())
nao_avaliavel = int((auto["imagem_avaliavel"].str.lower() == "false").sum())
print(f"\nerros: {erros}/{len(auto)} | não avaliáveis: {nao_avaliavel}")
if len(auto) and nao_avaliavel / len(auto) > 0.05:
    print(f"ATENÇÃO: {nao_avaliavel} fotos marcadas como não avaliáveis (>5%)")

if CSV_PATH.exists():
    hum = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
    if "ambiente" not in hum:
        hum["ambiente"] = "desconhecido"
    j = auto.merge(hum[["url_foto", "label", "ambiente"]], on="url_foto",
                   suffixes=("_auto", "_hum"))
    j = j[(j["erro"] == "") & (j["label_auto"] != "") & (j["label_hum"] != "")]

    if len(j):
        print(f"\n=== validação contra {len(j)} fotos rotuladas à mão ===")
        print("matriz de desgaste (linha = humano, coluna = agente):")
        print(pd.crosstab(j["label_hum"], j["label_auto"])
              .reindex(index=LABELS, columns=LABELS, fill_value=0).to_string())

        acuracia = (j["label_hum"] == j["label_auto"]).mean()
        kappa = _kappa_quadratico(j["label_hum"], j["label_auto"])
        print(f"\nacurácia: {acuracia:.1%} | kappa quadrático: {kappa:.3f}")
        print("\nrecall por classe (humano -> acerto):")
        for lab in LABELS:
            sub = j[j["label_hum"] == lab]
            if len(sub):
                print(f"  {lab:<16} {(sub['label_auto'] == lab).mean():.0%}  (n={len(sub)})")

        amb = j[j["ambiente_hum"].isin(["interna", "externa"])]
        if len(amb):
            print(f"\nambiente: {len(amb)} fotos | acurácia: "
                  f"{(amb['ambiente_hum'] == amb['ambiente_auto']).mean():.1%}")
            print(pd.crosstab(amb["ambiente_hum"], amb["ambiente_auto"]).to_string())

        if kappa >= 0.6 and acuracia >= 0.75:
            print("\n=> confiável para pré-rotular (revisão opcional)")
        elif kappa >= 0.4:
            print("\n=> use só como sugestão (revisão humana obrigatória)")
        else:
            print("\n=> NÃO use o desgaste automático; no máximo o ambiente")

        print("\n10 maiores divergências:")
        for _, r in j[j["label_hum"] != j["label_auto"]].head(10).iterrows():
            print(f"  humano={r['label_hum']:<16} agente={r['label_auto']:<16} "
                  f"conf={r['confianca']}  {r['defeitos'][:50]}")
    else:
        print("\nnenhuma foto em comum com o CSV humano ainda — rotule algumas para medir")

In [ ]:
# ============================================================================
# A/B — mesma amostra de 40 fotos nos 3 desenhos:
#   A: 400px + prompt "dê a nota"            (o que rodou no run v1)
#   B: 800px + prompt "dê a nota"
#   C: 800px + vistoriador (achados -> regra)     <- o que está na célula v3
#
# Usa OpenRouter direto (qwen3-vl-30b): o caminho estruturado da NVIDIA está
# instável (503) e o do OpenRouter não obedece schema. A resposta vem como
# texto JSON, parseada com json_repair. Só diagnóstico: o resultado NÃO entra
# no CSV de produção.
# ============================================================================

import asyncio
import base64
import io
import json
import os
import re
import time
from typing import Literal

import httpx
import pandas as pd
from IPython.display import HTML, display
from json_repair import repair_json
from openai import AsyncOpenAI
from pydantic import BaseModel, Field

AB_CSV = REPO / "dados" / "ab_conservacao.csv"
AB_SEED = 42
AB_POR_CLASSE = {"desgastado": 15, "precário": 5, "desgaste leve": 10, "bem conservado": 10}
AB_CONCURRENCY = 4
AB_TENTATIVAS = 4
AB_MODELO = "qwen/qwen3-vl-30b-a3b-instruct"
AB_MOSTRAR = True
AB_MOSTRAR_LARGURA = 320

AB_COLUNAS = ["variante", "url_foto", "label_humano", "label_auto", "defeitos",
              "justificativa", "confianca", "imagem_avaliavel", "modelo", "erro", "timestamp"]

AB_VARIANTES = [   # (variante, lado px, desenho)
    ("A_400_direto", 400, "direto"),
    ("B_800_direto", 800, "direto"),
    ("C_800_vistoriador", 800, "vistoriador"),
]


class ClassificacaoDireta(BaseModel):
    """Saída dos desenhos A/B: o modelo escolhe a nota diretamente."""
    label: Literal["bem conservado", "desgaste leve", "desgastado", "precário"]
    ambiente: Literal["interna", "externa", "ambigua"]
    justificativa: str
    confianca: float = Field(ge=0, le=1)


PROMPT_DIRETO = """Avalie o estado de conservação do imóvel nesta foto.
Escala (ordinal, escolha UMA):
- "bem conservado": acabamentos íntegros, limpo, sem sinais de uso
- "desgaste leve": sinais leves de uso (rejunte escuro, marcas pequenas, pintura encardida)
- "desgastado": desgaste evidente (manchas de umidade, pintura gasta, piso desgastado, rachaduras)
- "precário": dano grave (mofo, infiltração, rachadura estrutural, partes faltando)

Responda em JSON: {"label": ..., "ambiente": "interna"|"externa"|"ambigua",
"justificativa": "1 frase", "confianca": 0.0-1.0}"""


def _ab_url(url, lado):
    """Reescreve o resize do CDN; URLs sem dimension= caem no downscale local."""
    return re.sub(r"dimension=\d+x\d+", f"dimension={lado}x{lado}", url)


def _ab_imagem(url, lado):
    with httpx.stream("GET", _ab_url(url, lado), timeout=40, follow_redirects=True) as r:
        r.raise_for_status()
        bruto = b"".join(r.iter_bytes())
    from PIL import Image
    img = Image.open(io.BytesIO(bruto)).convert("RGB")
    if max(img.size) > lado:
        img.thumbnail((lado, lado))
    b = io.BytesIO()
    img.save(b, "JPEG", quality=60)
    return "data:image/jpeg;base64," + base64.b64encode(b.getvalue()).decode(), b.tell(), img.size


def _ab_ler():
    if not AB_CSV.exists():
        return []
    with open(AB_CSV, encoding="utf-8") as f:
        return list(pd.read_csv(f, dtype=str, keep_default_na=False).to_dict("records"))


def _ab_gravar(linhas):
    AB_CSV.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(linhas, columns=AB_COLUNAS).to_csv(AB_CSV, index=False, encoding="utf-8")


def _ab_amostra():
    hum = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
    pedacos = [g.sample(n=min(n, len(g)), random_state=AB_SEED)
               for lab, n in AB_POR_CLASSE.items()
               if len(g := hum[hum["label"] == lab])]
    s = pd.concat(pedacos).sample(frac=1, random_state=AB_SEED)
    print("amostra do A/B:", s["label"].value_counts().to_dict(), f"| {len(s)} fotos")
    return s


def _ab_cliente():
    key = os.getenv("OPENROUTER_API_KEY")
    if not key:
        raise RuntimeError("OPENROUTER_API_KEY ausente")
    return AsyncOpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)


def _ab_parse(txt, esquema):
    m = re.search(r"\{.*\}", txt or "", re.S)
    return esquema(**json.loads(repair_json(m.group(0) if m else (txt or ""))))


async def _ab_chamar(cliente, prompt, dataurl, esquema):
    resp = await cliente.chat.completions.create(
        model=AB_MODELO, temperature=0.1, max_tokens=1200,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": dataurl}}]}])
    return _ab_parse(resp.choices[0].message.content, esquema)


def _ab_linha(variante, url_foto, label_humano, saida, erro="", imagem_avaliavel=True):
    if erro:
        return dict(variante=variante, url_foto=url_foto, label_humano=label_humano,
                    label_auto="", defeitos="", justificativa="", confianca="",
                    imagem_avaliavel="", modelo=AB_MODELO, erro=erro[:200],
                    timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"))
    if isinstance(saida, AchadosFoto):
        label, achados = _derivar(saida)
        return dict(variante=variante, url_foto=url_foto, label_humano=label_humano,
                    label_auto=label, defeitos="; ".join(achados),
                    justificativa=saida.justificativa, confianca=f"{saida.confianca:.2f}",
                    imagem_avaliavel=str(saida.imagem_avaliavel).lower(),
                    modelo=AB_MODELO, erro="", timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"))
    return dict(variante=variante, url_foto=url_foto, label_humano=label_humano,
                label_auto=saida.label, defeitos="", justificativa=saida.justificativa,
                confianca=f"{saida.confianca:.2f}", imagem_avaliavel="true",
                modelo=AB_MODELO, erro="", timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"))


async def _ab_uma(cliente, sem, variante, lado, desenho, url, label_hum):
    async with sem:
        for t in range(1, AB_TENTATIVAS + 1):
            try:
                dataurl, nbytes, size = await asyncio.to_thread(_ab_imagem, url, lado)
                if desenho == "vistoriador":
                    s = await _ab_chamar(cliente, PROMPT_AUTO, dataurl, AchadosFoto)
                else:
                    s = await _ab_chamar(cliente, PROMPT_DIRETO, dataurl, ClassificacaoDireta)
                return _ab_linha(variante, url, label_hum, s), nbytes, size
            except Exception as e:  # noqa: BLE001
                if t == AB_TENTATIVAS:
                    return _ab_linha(variante, url, label_hum, None, f"{type(e).__name__}: {e}"), 0, (0, 0)
                await asyncio.sleep(2 * t)


async def rodar_ab():
    amostra = _ab_amostra()
    existentes = _ab_ler()
    feitas = {(r["variante"], r["url_foto"]) for r in existentes
              if not r.get("erro") and r.get("label_auto")}
    pend = [(v, lado, des, u, lab)
            for v, lado, des in AB_VARIANTES
            for u, lab in zip(amostra["url_foto"], amostra["label"])
            if (v, u) not in feitas]
    print(f"pendentes: {len(pend)} chamadas | já feitas: {len(feitas)}")

    cliente, sem = _ab_cliente(), asyncio.Semaphore(AB_CONCURRENCY)
    finais = []
    if pend:
        sem_preview, preview = asyncio.Semaphore(1), None
        if AB_MOSTRAR:
            preview = display(HTML("<i>rodando A/B…</i>"), display_id=True)
        for i, (v, lado, des, u, lab) in enumerate(pend, 1):
            r, nbytes, size = await _ab_uma(cliente, sem, v, lado, des, u, lab)
            finais.append(r)
            tag = f"{r['label_auto'] or r['erro'][:40]:<16}"
            print(f"{i:>3}/{len(pend)} {v:<18} {str(size):<12} {nbytes/1024:>5.0f}KB "
                  f"hum={lab:<16} auto={tag} {r['defeitos'][:35]}", flush=True)
            if preview is not None and getattr(preview, "update", None):
                with sem_preview:
                    try:
                        preview.update(HTML(
                            f'<img src="{u}" width="{AB_MOSTRAR_LARGURA}" style="border-radius:6px">'
                            f'<br><b>{v}</b>: {tag} <small>(hum: {lab})</small>'))
                    except Exception:
                        pass
            if i % 10 == 0:
                _ab_gravar(existentes + finais)
    _ab_gravar(existentes + finais)

    df = pd.DataFrame(existentes + finais, columns=AB_COLUNAS)
    ok = df[(df["erro"] == "") & (df["label_auto"] != "")]
    print(f"\nlinhas: {len(df)} | válidas: {len(ok)} | erros: {int((df['erro'] != '').sum())}")
    if not len(ok):
        return df

    print("\n=== distribuição prevista por variante ===")
    print(pd.crosstab(ok["variante"], ok["label_auto"])
          .reindex(columns=LABELS, fill_value=0).to_string())

    print("\n=== acurácia/kappa vs humano ===")
    for v, g in ok.groupby("variante"):
        acu = (g["label_auto"] == g["label_humano"]).mean()
        kap = _kappa_quadratico(g["label_humano"], g["label_auto"])
        print(f"  {v:<20} n={len(g):<3} acurácia={acu:.0%}  kappa={kap:.3f}")

    for v, g in ok.groupby("variante"):
        if v != AB_VARIANTES[-1][0]:
            continue
        print(f"\nmatriz {v} (linha = humano, coluna = agente):")
        print(pd.crosstab(g["label_humano"], g["label_auto"])
              .reindex(index=LABELS, columns=LABELS, fill_value=0).to_string())
        print("\nrecall por classe:")
        for lab in LABELS:
            sub = g[g["label_humano"] == lab]
            if len(sub):
                print(f"  {lab:<16} {(sub['label_auto'] == lab).mean():.0%} (n={len(sub)})")
    return df


if len(_ab_ler()) < len(AB_VARIANTES) * 40:
    await rodar_ab()
else:
    print("A/B já completo em", AB_CSV.name)

# Avaliando as classificações

Três leituras: **manual** (qualidade e equilíbrio do ground truth), **automática**
(cobertura, erros, defeitos) e **manual × automática** (acurácia, kappa quadrático,
matriz de confusão e recall por classe).

Kappa: 0 = acaso, 1 = perfeito. Para pré-rotular: ≥ 0,6 e acurácia ≥ 75%.
Abaixo de 0,4, o label automático não deve ser usado.

In [13]:
import pandas as pd
from pathlib import Path

LABELS = ["bem conservado", "desgaste leve", "desgastado", "precário"]

REPO = Path.cwd()
if not (REPO / "dados").exists():
    REPO = REPO.parent  # notebook aberto a partir de Notebooks/
CSV_PATH = REPO / "dados" / "rotulos_conservacao.csv"

hum = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
print(f"manual: {len(hum)} fotos | {hum['url_anuncio'].nunique()} anúncios")
if "amostra" in globals():
    total = len(hum) + len(amostra)
    print(f"progresso: {len(hum)}/{total} da amostra ({len(hum) / total:.0%})")

META = globals().get("META", {"bem conservado": 110, "desgaste leve": 370,
                              "desgastado": 295, "precário": 26})
print(f"\npor classe (meta {sum(META.values())}):")
for lab in LABELS:
    n = int((hum["label"] == lab).sum())
    print(f"  {lab:<16} {n:>3}  {n / len(hum):>5.0%}  meta {META[lab]:>3}  faltam {max(0, META[lab] - n):>3}")

print("\nambiente:", hum["ambiente"].value_counts().to_dict())
print("sem ambiente:", int((hum["ambiente"] == "desconhecido").sum()))
valor = pd.to_numeric(hum["valor_imovel"], errors="coerce")
print("\nvalor mediano por classe:")
for lab in LABELS:
    print(f"  {lab:<16} R$ {valor[hum['label'] == lab].median():>12,.0f}")

print("\nchecagens:")
print("  duplicadas:", int(hum["url_foto"].duplicated().sum()),
      "| label vazio:", int((hum["label"] == "").sum()),
      "| label fora da escala:", len(set(hum["label"]) - set(LABELS)))
print("  fotos por anúncio:", hum.groupby("url_anuncio").size().value_counts().to_dict())

manual: 377 fotos | 377 anúncios
progresso: 377/400 da amostra (94%)

por classe (meta 80):
  bem conservado    51    14%  faltam  29
  desgaste leve    175    46%  faltam   0
  desgastado       139    37%  faltam   0
  precário          12     3%  faltam  68

ambiente: {'interna': 237, 'externa': 140}
sem ambiente: 0

valor mediano por classe:
  bem conservado   R$    1,200,000
  desgaste leve    R$      750,000
  desgastado       R$      640,000
  precário         R$      724,500

checagens:
  duplicadas: 0 | label vazio: 0 | label fora da escala: 0
  fotos por anúncio: {1: 377}


In [14]:
from pathlib import Path

caminho = Path(globals().get("AUTO_CSV", REPO / "dados" / "rotulos_conservacao_auto_v2.csv"))
if not caminho.exists():
    candidatos = sorted(REPO.glob("dados/rotulos_conservacao_auto*.csv"),
                        key=lambda p: p.stat().st_mtime)
    caminho = candidatos[-1] if candidatos else None

if caminho is None or not caminho.exists():
    print("nenhum CSV automático — rode a célula auto-conservacao")
else:
    auto = pd.read_csv(caminho, dtype=str, keep_default_na=False)
    ok = auto[(auto["erro"] == "") & (auto["label"] != "")]
    print(f"arquivo: {caminho.name} | {len(auto)} linhas | "
          f"{auto['url_anuncio'].nunique()} anúncios")
    if "versao" in auto:
        print("versões:", auto["versao"].value_counts().to_dict())

    print(f"\nerros: {int((auto['erro'] != '').sum())} ({(auto['erro'] != '').mean():.1%})")
    if "imagem_avaliavel" in auto:
        print("não avaliáveis:", int((auto["imagem_avaliavel"].str.lower() == "false").sum()))

    print("\npor classe (só válidas):")
    for lab in LABELS:
        n = int((ok["label"] == lab).sum())
        print(f"  {lab:<16} {n:>3}  {n / len(ok):>5.0%}")

    if "confianca" in auto:
        conf = pd.to_numeric(auto["confianca"], errors="coerce")
        print(f"\nconfiança: mediana {conf.median():.2f} | "
              f"p10 {conf.quantile(.1):.2f} | p90 {conf.quantile(.9):.2f}")

    if "defeitos" in auto:
        freq = pd.Series([d for s in ok["defeitos"] for d in s.split("; ") if d]).value_counts()
        print("\ndefeitos:", freq.head(9).to_dict())
        print("média de defeitos/foto:", round(ok["defeitos"].apply(
            lambda s: len([x for x in s.split("; ") if x])).mean(), 2))
    if "evidencias" in auto:
        print("evidências vazias:", f"{(auto['evidencias'].str.strip() == '').mean():.1%}")
    if "timestamp" in auto:
        print("janela:", auto["timestamp"].min(), "→", auto["timestamp"].max())

arquivo: rotulos_conservacao_auto.csv | 479 linhas | 400 anúncios

erros: 33 (6.9%)
não avaliáveis: 28

por classe (só válidas):
  bem conservado   428    96%
  desgaste leve     10     2%
  desgastado         6     1%
  precário           2     0%

confiança: mediana 0.95 | p10 0.90 | p90 0.95
evidências vazias: 19.6%
janela: 2026-10-22T10:27:18 → 2026-10-22T13:46:48


In [15]:
if "auto" not in globals():
    print("rode a célula avaliacao-auto antes")
else:
    j = hum.merge(auto[["url_foto", "label", "ambiente", "justificativa", "confianca"]],
                  on="url_foto", suffixes=("_hum", "_auto"))
    j = j[(j["label_hum"] != "") & (j["label_auto"] != "")].copy()
    print(f"fotos comparáveis: {len(j)}\n")

    if not len(j):
        print("sem fotos em comum")
    else:
        print("matriz (linha = humano, coluna = agente):")
        print(pd.crosstab(j["label_hum"], j["label_auto"])
              .reindex(index=LABELS, columns=LABELS, fill_value=0).to_string())

        acu = (j["label_hum"] == j["label_auto"]).mean()
        kap = _kappa_quadratico(j["label_hum"], j["label_auto"])
        print(f"\nacurácia: {acu:.1%} | kappa quadrático: {kap:.3f}")

        print("\nrecall / precisão por classe:")
        for lab in LABELS:
            h_lab = j["label_hum"] == lab
            a_lab = j["label_auto"] == lab
            rec = (j.loc[h_lab, "label_auto"] == lab).mean() if h_lab.any() else float("nan")
            pre = (j.loc[a_lab, "label_hum"] == lab).mean() if a_lab.any() else float("nan")
            txt = lambda v: "  -" if pd.isna(v) else f"{v:>4.0%}"
            print(f"  {lab:<16} recall {txt(rec)}      precisão {txt(pre)}      (n={int(h_lab.sum())})")

        amb = j[j["ambiente_hum"].isin(["interna", "externa"])]
        if len(amb):
            print(f"\nambiente: {len(amb)} fotos | acurácia "
                  f"{(amb['ambiente_hum'] == amb['ambiente_auto']).mean():.0%}")

        boas = j[j["label_hum"] == "bem conservado"]
        ruins = j[j["label_hum"].isin(["desgastado", "precário"])]
        print(f"\nimpecáveis reconhecidos como tal: "
              f"{int((boas['label_auto'] == 'bem conservado').sum())}/{len(boas)}")
        print(f"acusa desgaste onde há: "
              f"{int(ruins['label_auto'].isin(['desgastado', 'precário']).sum())}/{len(ruins)}")

        print("\nmaiores divergências:")
        ordem = {c: i for i, c in enumerate(LABELS)}
        dist = j["label_hum"].map(ordem) - j["label_auto"].map(ordem)
        for i in dist.abs().sort_values(ascending=False).head(10).index:
            r = j.loc[i]
            print(f"  hum={r['label_hum']:<16} auto={r['label_auto']:<16} {r['justificativa'][:60]}")

        if kap >= 0.6 and acu >= 0.75:
            print("\n=> confiável para pré-rotular")
        elif kap >= 0.4:
            print("\n=> só como sugestão; revisão humana obrigatória")
        else:
            print("\n=> NÃO usar o desgaste automático")

fotos comparáveis: 344

matriz (linha = humano, coluna = agente):
label_auto      bem conservado  desgaste leve  desgastado  precário
label_hum                                                          
bem conservado              46              0           1         0
desgaste leve              156              0           0         0
desgastado                 123              5           1         0
precário                     6              4           2         0

acurácia: 13.7% | kappa quadrático: 0.041

recall / precisão por classe:
  bem conservado   recall  98%      precisão  14%      (n=47)
  desgaste leve    recall   0%      precisão   0%      (n=156)
  desgastado       recall   1%      precisão  25%      (n=129)
  precário         recall   0%      precisão   -      (n=12)

ambiente: 344 fotos | acurácia 87%

impecáveis reconhecidos como tal: 46/47
acusa desgaste onde há: 3/141

maiores divergências:
  hum=precário         auto=bem conservado   A casa apresenta paredes bra

In [16]:
from IPython.display import HTML, Image, display
import html as _html

FOTOS_POR_CLASSE = None     # None = todas
FOTOS_COLUNAS = 6
FOTOS_LARGURA = 190
FOTOS_USAR_IMAGEM = False   # True = lista vertical com display(Image), sem HTML

def _fmt_reais(v):
    try:
        return f"R$ {float(v) / 1e3:,.0f}k".replace(",", ".")
    except (TypeError, ValueError):
        return ""

def _img(url):
    u = _html.escape(str(url), quote=True)
    return (f'<img src="{u}" width="{FOTOS_LARGURA}" loading="lazy" '
            f'referrerpolicy="no-referrer">')

def _card(url, titulo, sub):
    return (f'<td style="padding:6px;vertical-align:top;width:{FOTOS_LARGURA + 12}px">'
            f'{_img(url)}<div style="font:12px/1.3 sans-serif">{titulo}<br>'
            f'<span style="color:#666">{sub}</span></div></td>')

def _galeria(cards):
    linhas = "".join("<tr>" + "".join(cards[i:i + FOTOS_COLUNAS]) + "</tr>"
                     for i in range(0, len(cards), FOTOS_COLUNAS))
    return f"<table>{linhas}</table>"

def _limitar(g):
    return g if not FOTOS_POR_CLASSE else g.head(FOTOS_POR_CLASSE)

if "hum" not in globals():
    hum = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)

print(f"manual: {len(hum)} fotos | exibindo: {'todas' if not FOTOS_POR_CLASSE else FOTOS_POR_CLASSE} por classe")
total = 0
for lab in LABELS:
    g = hum[hum["label"] == lab].copy()
    if not len(g):
        continue
    g["_v"] = pd.to_numeric(g["valor_imovel"], errors="coerce")
    g = _limitar(g.sort_values("_v", ascending=False, na_position="last"))
    total += len(g)
    print(f"  {lab}: {len(g)} fotos")
    if FOTOS_USAR_IMAGEM:
        for _, r in g.iterrows():
            display(Image(url=r["url_foto"], width=FOTOS_LARGURA))
            print(f"{lab} · {r['bairro']} · {_fmt_reais(r['valor_imovel'])} · {r['ambiente']}")
    else:
        cards = [_card(r["url_foto"], f"<b>{_html.escape(lab)}</b>",
                       f"{_html.escape(r['bairro'])} · {_fmt_reais(r['valor_imovel'])} · {_html.escape(r['ambiente'])}")
                 for _, r in g.iterrows()]
        display(HTML(f'<div style="margin:14px 0 2px"><b>{_html.escape(lab)}</b> — {len(g)} fotos</div>'
                     + _galeria(cards)))
print("total exibido:", total)

manual: 377 fotos | exibindo: todas por classe
  bem conservado: 51 fotos


  desgaste leve: 175 fotos


desgaste levegloria · R$ 14.000k · interna,desgaste leveanita garibaldi · R$ 8.900k · interna,desgaste levecentro · R$ 6.197k · interna,desgaste levesaguacu · R$ 5.800k · externa,desgaste levegloria · R$ 4.900k · interna,desgaste levesaguacu · R$ 4.850k · interna
desgaste leveboa vista · R$ 4.000k · interna,desgaste leveamerica · R$ 3.850k · interna,desgaste leveboa vista · R$ 3.850k · interna,desgaste leveatiradores · R$ 3.750k · interna,desgaste leveamerica · R$ 3.700k · externa,desgaste levepirabeiraba · R$ 3.200k · interna
desgaste levesanto antonio · R$ 3.199k · interna,desgaste leveamerica · R$ 2.995k · externa,desgaste leveboa vista · R$ 2.600k · externa,desgaste levecentro · R$ 2.590k · interna,desgaste leveatiradores · R$ 2.554k · interna,desgaste levevila nova · R$ 2.350k · interna
desgaste leveamerica · R$ 1.980k · interna,desgaste levecentro · R$ 1.921k · externa,desgaste leveamerica · R$ 1.900k · interna,desgaste levevila nova · R$ 1.900k · interna,desgaste levecentro · R$ 1.795k · interna,desgaste leveanita garibaldi · R$ 1.700k · interna
desgaste levecentro · R$ 1.700k · interna,desgaste levecosta e silva · R$ 1.690k · interna,desgaste leveamerica · R$ 1.600k · interna,desgaste leveanita garibaldi · R$ 1.600k · externa,desgaste levebom retiro · R$ 1.560k · interna,desgaste leveatiradores · R$ 1.530k · interna
desgaste leveamerica · R$ 1.529k · externa,desgaste leveanita garibaldi · R$ 1.517k · interna,desgaste leveanita garibaldi · R$ 1.500k · interna,desgaste levesaguacu · R$ 1.499k · interna,desgaste leveamerica · R$ 1.490k · interna,desgaste levepirabeiraba · R$ 1.450k · interna
desgaste levesaguacu · R$ 1.450k · externa,desgaste leveamerica · R$ 1.430k · externa,desgaste levegloria · R$ 1.400k · interna,desgaste leveamerica · R$ 1.390k · externa,desgaste levebom retiro · R$ 1.390k · interna,desgaste leveanita garibaldi · R$ 1.380k · externa
desgaste leveatiradores · R$ 1.350k · interna,desgaste leveanita garibaldi · R$ 1.350k · interna,desgaste leveatiradores · R$ 1.350k · interna,desgaste leveamerica · R$ 1.329k · interna,desgaste levecosta e silva · R$ 1.259k · interna,desgaste leveatiradores · R$ 1.250k · interna
desgaste levegloria · R$ 1.250k · interna,desgaste levebucarein · R$ 1.200k · interna,desgaste levebom retiro · R$ 1.197k · interna,desgaste levecosta e silva · R$ 1.140k · interna,desgaste leveatiradores · R$ 1.137k · interna,desgaste levesaguacu · R$ 1.125k · externa
desgaste leveanita garibaldi · R$ 1.117k · interna,desgaste leveamerica · R$ 1.060k · interna,desgaste leveatiradores · R$ 1.040k · interna,desgaste levebucarein · R$ 1.000k · interna,desgaste leveiririu · R$ 995k · interna,desgaste leveboa vista · R$ 990k · interna
desgaste levebucarein · R$ 990k · interna,desgaste levejardim sofia · R$ 989k · interna,desgaste levejardim sofia · R$ 980k · externa,desgaste levebom retiro · R$ 979k · externa,desgaste leveamerica · R$ 959k · externa,desgaste levecomasa · R$ 950k · interna


  desgastado: 139 fotos


desgastadoamerica · R$ 2.950k · externa,desgastadoanita garibaldi · R$ 2.500k · externa,desgastadogloria · R$ 2.484k · interna,desgastadosaguacu · R$ 2.300k · interna,desgastadoitinga · R$ 2.250k · externa,desgastadoamerica · R$ 2.239k · externa
desgastadosao marcos · R$ 2.000k · interna,desgastadoitaum · R$ 1.890k · interna,desgastadoanita garibaldi · R$ 1.690k · interna,desgastadoanita garibaldi · R$ 1.600k · interna,desgastadoatiradores · R$ 1.530k · interna,desgastadopirabeiraba · R$ 1.500k · externa
desgastadoiririu · R$ 1.400k · interna,desgastadoamerica · R$ 1.390k · interna,desgastadoaventureiro · R$ 1.390k · interna,desgastadosanto antonio · R$ 1.390k · externa,desgastadoanita garibaldi · R$ 1.350k · interna,desgastadovila nova · R$ 1.300k · interna
desgastadoparque guarani · R$ 1.300k · externa,desgastadocentro · R$ 1.272k · interna,desgastadoatiradores · R$ 1.250k · interna,desgastadobom retiro · R$ 1.200k · interna,desgastadoanita garibaldi · R$ 1.200k · externa,desgastadofloresta · R$ 1.200k · externa
desgastadoanita garibaldi · R$ 1.165k · interna,desgastadoanita garibaldi · R$ 1.150k · externa,desgastadoamerica · R$ 1.100k · interna,desgastadoatiradores · R$ 1.100k · externa,desgastadocentro · R$ 1.050k · interna,desgastadoatiradores · R$ 999k · interna
desgastadosaguacu · R$ 995k · interna,desgastadosanto antonio · R$ 990k · interna,desgastadosaguacu · R$ 960k · interna,desgastadofloresta · R$ 950k · interna,desgastadosaguacu · R$ 950k · interna,desgastadojardim iririu · R$ 950k · interna
desgastadofloresta · R$ 930k · externa,desgastadocomasa · R$ 900k · interna,desgastadofatima · R$ 900k · externa,desgastadocomasa · R$ 898k · externa,desgastadoamerica · R$ 890k · interna,desgastadosaguacu · R$ 890k · interna
desgastadonova brasilia · R$ 890k · interna,desgastadosanto antonio · R$ 890k · externa,desgastadoatiradores · R$ 880k · externa,desgastadoadhemar garcia · R$ 850k · interna,desgastadovila nova · R$ 850k · externa,desgastadopetropolis · R$ 840k · interna
desgastadopetropolis · R$ 839k · interna,desgastadocosta e silva · R$ 835k · interna,desgastadoiririu · R$ 830k · interna,desgastadoguanabara · R$ 829k · interna,desgastadofatima · R$ 820k · interna,desgastadoboa vista · R$ 797k · interna
desgastadogloria · R$ 795k · interna,desgastadoiririu · R$ 790k · interna,desgastadoanita garibaldi · R$ 780k · externa,desgastadosao marcos · R$ 780k · externa,desgastadofloresta · R$ 760k · interna,desgastadosanto antonio · R$ 750k · interna
desgastadoparanaguamirim · R$ 740k · externa,desgastadoiririu · R$ 730k · interna,desgastadovila nova · R$ 705k · externa,desgastadocosta e silva · R$ 695k · interna,desgastadosanto antonio · R$ 695k · interna,desgastadoaventureiro · R$ 688k · externa


  precário: 12 fotos


precárioamerica · R$ 1.380k · externa,precáriogloria · R$ 1.200k · externa,precáriosaguacu · R$ 1.200k · interna,precáriocosta e silva · R$ 817k · interna,precáriocosta e silva · R$ 785k · interna,precárioiririu · R$ 749k · externa
precáriofloresta · R$ 700k · externa,precáriobucarein · R$ 585k · externa,precáriofloresta · R$ 350k · externa,precárioguanabara · R$ 350k · externa,precárioaventureiro · R$ 339k · interna,precárioulisses guimaraes · R$ 259k · interna


total exibido: 377


In [17]:
from IPython.display import HTML, Image, display
import html as _html

FOTOS_DIVERG_MAX = None     # None = todas
FOTOS_COLUNAS = 6
FOTOS_LARGURA = 190
FOTOS_USAR_IMAGEM = False

if "_card" not in globals() or "_galeria" not in globals():
    raise RuntimeError("rode a célula fotos-manual antes")

if "auto" not in globals():
    print("rode avaliacao-auto antes")
else:
    if "j" not in globals() or not len(globals()["j"]):
        j = hum.merge(auto[["url_foto", "label", "ambiente", "justificativa", "confianca"]],
                      on="url_foto", suffixes=("_hum", "_auto"))
        j = j[(j["label_hum"] != "") & (j["label_auto"] != "")].copy()
    ordem = {c: i for i, c in enumerate(LABELS)}
    g = j.assign(_d=(j["label_hum"].map(ordem) - j["label_auto"].map(ordem)).abs())
    g = g.sort_values(["_d", "label_hum"], ascending=[False, True])
    if FOTOS_DIVERG_MAX:
        g = g.head(FOTOS_DIVERG_MAX)
    n_div = int((g["label_hum"] != g["label_auto"]).sum())
    print(f"comparáveis: {len(j)} | exibindo: {len(g)} | divergências: {n_div}")
    if not len(g):
        print("sem fotos em comum")
    elif FOTOS_USAR_IMAGEM:
        for _, r in g.iterrows():
            display(Image(url=r["url_foto"], width=FOTOS_LARGURA))
            print(f"hum: {r['label_hum']} | auto: {r['label_auto']} | "
                  f"{r['bairro']} | conf {r.get('confianca', '')}")
    else:
        cards = []
        for _, r in g.iterrows():
            dif = r["label_hum"] != r["label_auto"]
            marca = '<b style="color:#c62828">DIFERE</b>' if dif else "<b>ok</b>"
            cards.append(_card(
                r["url_foto"],
                f"hum <b>{_html.escape(r['label_hum'])}</b><br>"
                f"auto {_html.escape(r['label_auto'] or '-')} · {marca}",
                f"{_html.escape(r['bairro'])} · conf {_html.escape(str(r.get('confianca', '')))}"))
        display(HTML(f'<div style="margin:14px 0 2px"><b>manual × automático</b> — {len(g)} fotos '
                     f'(<span style="color:#c62828">{n_div} divergem</span>)</div>' + _galeria(cards)))

comparáveis: 344 | exibindo: 344 | divergências: 297


hum precárioauto bem conservado · DIFEREgloria · conf 0.90,hum precárioauto bem conservado · DIFEREbucarein · conf 0.90,hum precárioauto bem conservado · DIFEREguanabara · conf 0.90,hum precárioauto bem conservado · DIFEREiririu · conf 0.20,hum precárioauto bem conservado · DIFEREfloresta · conf 0.80,hum precárioauto bem conservado · DIFEREulisses guimaraes · conf 0.90
hum bem conservadoauto desgastado · DIFEREbom retiro · conf 0.85,hum desgastadoauto bem conservado · DIFEREjarivatuba · conf 0.90,hum desgastadoauto bem conservado · DIFEREfatima · conf 0.90,hum desgastadoauto bem conservado · DIFEREgloria · conf 0.95,hum desgastadoauto bem conservado · DIFEREpetropolis · conf 0.95,hum desgastadoauto bem conservado · DIFEREaventureiro · conf 0.90
hum desgastadoauto bem conservado · DIFEREcosta e silva · conf 0.95,hum desgastadoauto bem conservado · DIFEREfloresta · conf 0.90,hum desgastadoauto bem conservado · DIFEREamerica · conf 0.90,hum desgastadoauto bem conservado · DIFEREsaguacu · conf 0.90,hum desgastadoauto bem conservado · DIFEREbom retiro · conf 0.90,hum desgastadoauto bem conservado · DIFEREjarivatuba · conf 0.90
hum desgastadoauto bem conservado · DIFEREjardim iririu · conf 0.95,hum desgastadoauto bem conservado · DIFEREsaguacu · conf 0.96,hum desgastadoauto bem conservado · DIFEREsaguacu · conf 0.90,hum desgastadoauto bem conservado · DIFEREjardim paraiso · conf 0.90,hum desgastadoauto bem conservado · DIFEREgloria · conf 0.95,hum desgastadoauto bem conservado · DIFEREgloria · conf 0.95
hum desgastadoauto bem conservado · DIFEREprofipo · conf 0.95,hum desgastadoauto bem conservado · DIFEREvila nova · conf 0.96,hum desgastadoauto bem conservado · DIFEREguanabara · conf 0.90,hum desgastadoauto bem conservado · DIFEREanita garibaldi · conf 0.95,hum desgastadoauto bem conservado · DIFEREguanabara · conf 0.95,hum desgastadoauto bem conservado · DIFEREcomasa · conf 0.95
hum desgastadoauto bem conservado · DIFEREboehmerwald · conf 0.95,hum desgastadoauto bem conservado · DIFEREsaguacu · conf 0.90,hum desgastadoauto bem conservado · DIFEREitaum · conf 0.95,hum desgastadoauto bem conservado · DIFEREsanto antonio · conf 0.95,hum desgastadoauto bem conservado · DIFEREcosta e silva · conf 0.95,hum desgastadoauto bem conservado · DIFEREgloria · conf 0.95
hum desgastadoauto bem conservado · DIFEREvila nova · conf 0.90,hum desgastadoauto bem conservado · DIFEREsaguacu · conf 0.95,hum desgastadoauto bem conservado · DIFEREsanto antonio · conf 0.95,hum desgastadoauto bem conservado · DIFEREatiradores · conf 0.90,hum desgastadoauto bem conservado · DIFEREamerica · conf 0.95,hum desgastadoauto bem conservado · DIFEREsaguacu · conf 0.90
hum desgastadoauto bem conservado · DIFEREatiradores · conf 0.80,hum desgastadoauto bem conservado · DIFEREfloresta · conf 0.95,hum desgastadoauto bem conservado · DIFEREanita garibaldi · conf 0.95,hum desgastadoauto bem conservado · DIFEREcentro · conf 0.95,hum desgastadoauto bem conservado · DIFEREparanaguamirim · conf 0.90,hum desgastadoauto bem conservado · DIFEREsanto antonio · conf 0.90
hum desgastadoauto bem conservado · DIFEREpetropolis · conf 0.90,hum desgastadoauto bem conservado · DIFEREcosta e silva · conf 0.90,hum desgastadoauto bem conservado · DIFEREvila nova · conf 0.90,hum desgastadoauto bem conservado · DIFEREfloresta · conf 0.90,hum desgastadoauto bem conservado · DIFEREanita garibaldi · conf 0.95,hum desgastadoauto bem conservado · DIFEREcentro · conf 0.90
hum desgastadoauto bem conservado · DIFEREatiradores · conf 0.95,hum desgastadoauto bem conservado · DIFEREvila nova · conf 0.95,hum desgastadoauto bem conservado · DIFEREcosta e silva · conf 0.20,hum desgastadoauto bem conservado · DIFEREjoao costa · conf 0.90,hum desgastadoauto bem conservado · DIFEREitaum · conf 0.85,hum desgastadoauto bem conservado · DIFEREbom retiro · conf 0.95
hum desgastadoauto bem conservado · DIFEREaventureiro · conf 0.85,hum desgastadoauto bem conservado · DIFEREfloresta · conf 0.90,hum desgastadoauto be